# 03 — Evaluating the sales intelligence agent

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/02_sales_intelligence/03_evaluate.ipynb)

**Prerequisites**:
- [02_build.ipynb](./02_build.ipynb) — The complete sales intelligence pipeline
- Basic understanding of evaluation metrics and statistical testing

**What you will learn**:
- How to use LLM-as-a-judge to score email quality on multiple dimensions
- How to measure personalization depth beyond surface-level metrics
- How to build a statistically rigorous A/B testing framework for outreach
- How to compute precision / recall for response classification
- How to calculate the cost and ROI of an AI-powered sales pipeline

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "matplotlib>=3.7.0" "scipy>=1.10.0" "numpy>=1.24.0"

import os
import json
import random
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

random.seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True})
print("Setup complete.")

## Section 1 — Evaluation framework

Before measuring anything we need to define *what good looks like*.
A sales intelligence agent has multiple outputs, each requiring its
own metric family:

| Component | Metric | How measured |
|-----------|--------|--------------|
| Email quality | Personalization, relevance, professionalism, CTA clarity | LLM-as-judge (1-5) |
| Research | Accuracy & coverage of hooks | Manual spot-check + LLM |
| Lead scoring | Precision at top-k | Compare with ground-truth tiers |
| Response classifier | Precision, recall per category | Held-out labeled set |
| Overall pipeline | Cost per prospect, time savings, ROI | Token usage + timing |

We implement each evaluation in the sections that follow.

In [ ]:
# Simulated generated emails to evaluate (10 emails)
GENERATED_EMAILS = [
    {
        "prospect": "NovaPay", "industry": "FinTech",
        "subject": "Cutting NovaPay's deploy time after your Series B",
        "body": ("Hi Alex, congrats on NovaPay's Series B with Sequoia! "
                 "With 8 new backend engineers joining, I imagine CI/CD pipeline "
                 "bottlenecks are top of mind. We helped TrustVault (similar Kafka + "
                 "PostgreSQL stack) cut deploy cycles from 40 min to 14 min. "
                 "Worth a 15-min call to see if that's relevant?"),
        "hooks": ["Series B funding", "hiring backend engineers", "Kafka stack"],
    },
    {
        "prospect": "MedSync", "industry": "HealthTech",
        "subject": "MedSync's FHIR migration and deployment velocity",
        "body": ("Hi Sam, I saw MedSync published a blog on migrating to FHIR — "
                 "that's a heavy lift. Teams in similar transitions often see deploy "
                 "frequency drop 30% mid-migration. DevPipe kept CareLink shipping "
                 "daily through their FHIR rollout. Interested in a quick walkthrough?"),
        "hooks": ["FHIR migration blog", "deploy frequency risk"],
    },
    {
        "prospect": "CodeLoom", "industry": "SaaS / DevTools",
        "subject": "Scaling CodeLoom's pipeline for 12 new engineers",
        "body": ("Hi Jordan, noticed CodeLoom is hiring 12 senior backend engineers "
                 "— exciting growth! With a Go + Kubernetes stack, keeping build times "
                 "under control at that scale is tricky. We helped BuildKite (also Go/K8s) "
                 "maintain sub-5-min builds at 200 engineers. Want to see how?"),
        "hooks": ["hiring 12 engineers", "Go + Kubernetes stack", "build times"],
    },
    {
        "prospect": "CartVibe", "industry": "E-Commerce",
        "subject": "CartVibe's Shopify integration pipeline",
        "body": ("Hi Taylor, saw CartVibe expanded into EMEA — congrats! "
                 "Shopify API integrations across regions often mean more deploys "
                 "and more pipeline flakiness. DevPipe reduced ShopNova's failed "
                 "deploys by 45% during their APAC launch. Quick chat?"),
        "hooks": ["EMEA expansion", "Shopify API", "pipeline reliability"],
    },
    {
        "prospect": "LedgerAI", "industry": "FinTech",
        "subject": "LedgerAI: faster deploys after your new CTO hire",
        "body": ("Hi Morgan, congrats on bringing Jordan Lee on as CTO from Stripe! "
                 "New engineering leadership often means a fresh look at CI/CD. "
                 "DevPipe helped PayGrid's new CTO cut deploy friction by 60% in "
                 "the first quarter. Worth exploring?"),
        "hooks": ["new CTO from Stripe", "leadership transition"],
    },
    {
        "prospect": "PulseHealth", "industry": "HealthTech",
        "subject": "PulseHealth's hiring surge and deploy velocity",
        "body": ("Hi Alex, PulseHealth is hiring ML engineers for the platform team — "
                 "sounds like a big push. Scaling deploys while onboarding ML "
                 "workloads is a common pain point. We helped GenoWell handle "
                 "3x deploy volume without adding DevOps headcount. Interested?"),
        "hooks": ["ML engineer hiring", "platform team scaling"],
    },
    {
        "prospect": "ShipStack", "industry": "SaaS / DevTools",
        "subject": "ShipStack — case study on Rust + gRPC pipelines",
        "body": ("Hi Sam, I noticed ShipStack uses Rust and gRPC — not many "
                 "CI/CD tools handle that stack well. We built first-class Rust "
                 "support after working with PipeForge on exactly this. "
                 "Want me to send the architecture doc?"),
        "hooks": ["Rust + gRPC stack", "niche tooling gap"],
    },
    {
        "prospect": "BuyBetter", "industry": "E-Commerce",
        "subject": "BuyBetter: deploy confidence during peak season",
        "body": ("Hi Jordan, with BuyBetter's new analytics product line for "
                 "mid-market, I'd guess deploy confidence is critical — especially "
                 "heading into peak season. DevPipe gave StyleLoop zero-downtime "
                 "deploys through Black Friday. Quick 15 min?"),
        "hooks": ["new analytics product", "peak season deploys"],
    },
    {
        "prospect": "TrustVault", "industry": "FinTech",
        "subject": "TrustVault: compliance-friendly CI/CD",
        "body": ("Hi Taylor, FinTech compliance and fast deploys don't usually "
                 "mix. We built audit-trail pipelines for CompliBot that passed "
                 "SOC 2 on the first try. Worth a look for TrustVault?"),
        "hooks": ["FinTech compliance", "SOC 2 audit trails"],
    },
    {
        "prospect": "CareLink", "industry": "HealthTech",
        "subject": "CareLink's AWS Lambda scaling and deploy speed",
        "body": ("Hi Morgan, CareLink's TypeScript + Lambda stack is great for "
                 "velocity — until cold starts and deploy sizes creep up. "
                 "We helped HealthBridge keep Lambda deploys under 30 sec at "
                 "500 functions. Shall I share the playbook?"),
        "hooks": ["TypeScript + Lambda stack", "cold start issues", "deploy sizes"],
    },
]

print(f"Evaluation corpus: {len(GENERATED_EMAILS)} emails across "
      f"{len(set(e['industry'] for e in GENERATED_EMAILS))} industries")
df_emails = pd.DataFrame(GENERATED_EMAILS)
print(df_emails[["prospect", "industry", "subject"]].to_string(index=False))

## Section 2 — Email quality (LLM-as-judge)

Manual review doesn't scale. Instead we use a second LLM as an
evaluator — the **LLM-as-judge** pattern. For each email the judge
scores four dimensions on a 1-5 scale:

| Dimension | 1 (poor) | 5 (excellent) |
|-----------|----------|---------------|
| Personalization | Generic / mail-merge only | References 3+ specific company facts |
| Relevance | No connection to prospect's situation | Directly addresses an active pain point |
| Professionalism | Typos, pushy, or spammy tone | Polished, concise, respectful |
| CTA clarity | No ask / vague ask | Specific, low-friction next step |

In [ ]:
JUDGE_SYSTEM = """You are an expert email quality evaluator for B2B sales outreach.
Score the email on four dimensions (1-5 each). Be strict — most emails are 2-3.

Output JSON:
{
  "personalization": <int 1-5>,
  "relevance": <int 1-5>,
  "professionalism": <int 1-5>,
  "cta_clarity": <int 1-5>,
  "reasoning": "<1-2 sentence justification>"
}"""

def judge_email(email: dict) -> dict:
    """Score a single email using LLM-as-judge."""
    user_msg = json.dumps({
        "prospect_company": email["prospect"],
        "industry": email["industry"],
        "subject": email["subject"],
        "body": email["body"],
        "hooks_claimed": email["hooks"],
    }, indent=2)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_msg},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
    )
    return json.loads(response.choices[0].message.content)

# Score all 10 emails
scores: list[dict] = []
for email in GENERATED_EMAILS:
    result = judge_email(email)
    result["prospect"] = email["prospect"]
    scores.append(result)

df_scores = pd.DataFrame(scores)
dims = ["personalization", "relevance", "professionalism", "cta_clarity"]
print(df_scores[["prospect"] + dims].to_string(index=False))
print(f"\nAverages:")
for d in dims:
    print(f"  {d:<20s}: {df_scores[d].mean():.2f} / 5")
print(f"  {'overall':<20s}: {df_scores[dims].mean(axis=1).mean():.2f} / 5")

## Section 3 — Personalization depth analysis

Not all personalization is equal. We classify every hook used into
three tiers and measure how often the pipeline reaches the deepest
level:

| Tier | Examples | Reply-rate impact |
|------|----------|-------------------|
| **Surface** | Uses first name, company name | +1 % |
| **Contextual** | References recent news, hiring, funding | +5 % |
| **Deep** | Infers pain from tech stack + growth stage | +12 % |

A high-quality pipeline should have >50 % of hooks at the contextual
or deep level.

In [ ]:
DEPTH_SYSTEM = """Classify each personalization hook into one of three depth tiers.
- surface: uses only name, title, or company name
- contextual: references a specific, verifiable event (news, hiring, funding)
- deep: infers a pain point or need not explicitly stated in public data

Input: JSON list of hooks.
Output: JSON {"hooks": [{"hook": "...", "tier": "surface|contextual|deep"}, ...]}."""

def classify_hooks(hooks: list[str]) -> list[dict]:
    """Classify a list of hooks by personalization depth."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": DEPTH_SYSTEM},
            {"role": "user", "content": json.dumps(hooks)},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
    )
    data = json.loads(response.choices[0].message.content)
    return data if isinstance(data, list) else data.get("hooks", data.get("results", []))

# Analyse all hooks across 10 emails
all_classified: list[dict] = []
for email in GENERATED_EMAILS:
    classified = classify_hooks(email["hooks"])
    for item in classified:
        item["prospect"] = email["prospect"]
    all_classified.extend(classified)

df_hooks = pd.DataFrame(all_classified)
tier_counts = df_hooks["tier"].value_counts()
total = len(df_hooks)

print("Hook depth distribution:")
for tier in ["surface", "contextual", "deep"]:
    count = tier_counts.get(tier, 0)
    pct = count / total * 100
    bar = "\u2588" * int(pct / 2)
    print(f"  {tier:<12s}: {count:>3d} ({pct:5.1f}%)  {bar}")

deep_pct = (tier_counts.get("contextual", 0) + tier_counts.get("deep", 0)) / total * 100
print(f"\nContextual + Deep: {deep_pct:.1f}% \u2014 "
      f"{'\u2705 PASS' if deep_pct >= 50 else '\u274c BELOW 50% TARGET'}")

## Section 4 — A/B testing framework

Even when emails *look* good to an LLM judge, the only metric that
matters is **reply rate in production**. We need a rigorous A/B
testing framework to compare variants.

This section builds three tools:
1. **Simulated send data** — generate realistic open / reply events
2. **Significance test** — two-proportion z-test via `scipy.stats`
3. **Power analysis** — compute the minimum sample size for a given
   minimum detectable effect (MDE)

In [ ]:
def simulate_campaign(
    n_per_variant: int = 2000,
    rate_a: float = 0.035,
    rate_b: float = 0.055,
    seed: int = 42,
) -> pd.DataFrame:
    """Simulate A/B send data with known reply rates."""
    rng = np.random.default_rng(seed)
    replies_a = rng.binomial(1, rate_a, n_per_variant)
    replies_b = rng.binomial(1, rate_b, n_per_variant)
    df = pd.DataFrame({
        "variant": ["A"] * n_per_variant + ["B"] * n_per_variant,
        "replied": np.concatenate([replies_a, replies_b]),
    })
    return df

def two_proportion_z_test(
    successes_a: int, n_a: int,
    successes_b: int, n_b: int,
) -> dict:
    """Run a two-proportion z-test and return z-stat, p-value, and lift."""
    p_a = successes_a / n_a
    p_b = successes_b / n_b
    p_pool = (successes_a + successes_b) / (n_a + n_b)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se if se > 0 else 0.0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    lift = (p_b - p_a) / p_a if p_a > 0 else float("inf")
    return {"z": round(z, 3), "p_value": round(p_value, 4), "lift_pct": round(lift * 100, 1)}

def required_sample_size(
    baseline_rate: float = 0.035,
    mde: float = 0.015,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """Compute required N per variant for a two-proportion test."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p1 = baseline_rate
    p2 = baseline_rate + mde
    numerator = (z_alpha * np.sqrt(2 * p1 * (1 - p1)) +
                 z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    denominator = (p2 - p1) ** 2
    return int(np.ceil(numerator / denominator))

# Run simulation
df_ab = simulate_campaign(n_per_variant=2000, rate_a=0.035, rate_b=0.055)
summary = df_ab.groupby("variant")["replied"].agg(["sum", "count", "mean"])
summary.columns = ["replies", "sends", "rate"]
print("Campaign simulation results:")
print(summary.to_string())

sa, na = int(summary.loc["A", "replies"]), int(summary.loc["A", "sends"])
sb, nb = int(summary.loc["B", "replies"]), int(summary.loc["B", "sends"])
result = two_proportion_z_test(sa, na, sb, nb)
print(f"\nZ-test: z={result['z']}, p={result['p_value']}, lift={result['lift_pct']}%")
sig = "\u2705 YES" if result["p_value"] < 0.05 else "\u274c NO"
print(f"Significant at \u03b1=0.05? {sig}")

# Power analysis table
print(f"\n{'MDE':>6s}  {'N per variant':>14s}")
print("-" * 24)
for mde in [0.005, 0.010, 0.015, 0.020, 0.030]:
    n = required_sample_size(baseline_rate=0.035, mde=mde)
    print(f"{mde:>6.3f}  {n:>14,}")

## Section 5 — Response classification accuracy

We evaluate the classifier from `02_build.ipynb` on a held-out
labeled test set. For each category we compute precision, recall,
and F1, then visualise the confusion matrix.

In [ ]:
# Held-out labeled test set (20 examples)
HELD_OUT = [
    ("Great, let's chat tomorrow at 10 AM.", "positive"),
    ("I'd love to learn more — send a Calendly?", "positive"),
    ("This could work. Can you loop in my team?", "positive"),
    ("Our budget is frozen until next fiscal year.", "objection_price"),
    ("That's more than we'd spend on this category.", "objection_price"),
    ("We're mid-migration, can't take on anything new until June.", "objection_timing"),
    ("Circle back in Q4 when we start planning.", "objection_timing"),
    ("Not my area — our VP Sales handles tooling.", "objection_authority"),
    ("You should email our engineering director.", "objection_authority"),
    ("Let me forward this to the person who owns CI/CD.", "objection_authority"),
    ("No thanks, we're all set.", "not_interested"),
    ("Please don't contact me again.", "not_interested"),
    ("We're not in the market for this.", "not_interested"),
    ("Out of office until March 20. Back on the 21st.", "out_of_office"),
    ("I'm traveling with limited email — back April 1.", "out_of_office"),
    ("550 5.1.1 User unknown.", "bounce"),
    ("Mail delivery failed: returning message to sender.", "bounce"),
    ("This is an automated reply: I am on leave.", "out_of_office"),
    ("We signed with a competitor last month.", "not_interested"),
    ("Interesting timing — we just started evaluating tools!", "positive"),
]

CATEGORIES = [
    "positive", "objection_price", "objection_timing",
    "objection_authority", "not_interested", "out_of_office", "bounce",
]

CLASSIFY_SYSTEM = """You are a sales response classifier. Classify the prospect's
reply into exactly one category. Respond with JSON: {"category": "...", "confidence": 0.0-1.0}

Categories: positive, objection_price, objection_timing, objection_authority,
not_interested, out_of_office, bounce."""

FEW_SHOT_EXAMPLES = [
    ("Sure, I'd love to chat. How about Thursday?", "positive"),
    ("This is way out of our budget right now.", "objection_price"),
    ("We're in a code freeze until Q2, bad timing.", "objection_timing"),
    ("I'm not the decision maker — try our VP Eng.", "objection_authority"),
    ("Not interested, please don't email me again.", "not_interested"),
    ("I'm out of office until March 15.", "out_of_office"),
    ("550 User unknown.", "bounce"),
]

def classify_response(reply_text: str) -> str:
    """Classify a prospect reply and return the predicted category."""
    few_shot_msgs: list[dict] = []
    for text, cat in FEW_SHOT_EXAMPLES:
        few_shot_msgs.append({"role": "user", "content": text})
        few_shot_msgs.append({"role": "assistant",
                              "content": json.dumps({"category": cat, "confidence": 0.95})})
    messages = [{"role": "system", "content": CLASSIFY_SYSTEM}] + few_shot_msgs + [
        {"role": "user", "content": reply_text}
    ]
    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        response_format={"type": "json_object"}, temperature=0.0,
    )
    return json.loads(response.choices[0].message.content).get("category", "unknown")

# Classify held-out set
y_true = [label for _, label in HELD_OUT]
y_pred = [classify_response(text) for text, _ in HELD_OUT]

# Per-category metrics
print(f"{'Category':<25s} {'Prec':>6s} {'Recall':>6s} {'F1':>6s} {'Support':>7s}")
print("-" * 52)
for cat in CATEGORIES:
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == cat and p == cat)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != cat and p == cat)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == cat and p != cat)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    support = sum(1 for t in y_true if t == cat)
    print(f"{cat:<25s} {prec:>6.2f} {rec:>6.2f} {f1:>6.2f} {support:>7d}")

accuracy = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)
print(f"\nOverall accuracy: {accuracy:.1%}")

In [ ]:
# Confusion matrix visualization
cat_to_idx = {c: i for i, c in enumerate(CATEGORIES)}
matrix = np.zeros((len(CATEGORIES), len(CATEGORIES)), dtype=int)
for t, p in zip(y_true, y_pred):
    ti = cat_to_idx.get(t, 0)
    pi = cat_to_idx.get(p, 0)
    matrix[ti][pi] += 1

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix, cmap="Blues")
ax.set_xticks(range(len(CATEGORIES)))
ax.set_yticks(range(len(CATEGORIES)))
short_labels = [c.replace("objection_", "obj_") for c in CATEGORIES]
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Response Classification — Confusion Matrix")

for i in range(len(CATEGORIES)):
    for j in range(len(CATEGORIES)):
        val = matrix[i][j]
        color = "white" if val > matrix.max() / 2 else "black"
        ax.text(j, i, str(val), ha="center", va="center", color=color, fontsize=11)

fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Section 6 — Cost and efficiency

AI-powered outreach only makes sense if the unit economics work.
We estimate three key figures:

1. **Cost per prospect** — API token costs for research + scoring + email + classification
2. **Time savings** — hours saved vs. a human SDR performing the same tasks
3. **ROI** — net revenue impact compared to the fully loaded cost of the pipeline

In [ ]:
# Cost model (GPT-4o-mini pricing as of mid-2024)
INPUT_COST_PER_1K = 0.000150   # $0.150 per 1M input tokens
OUTPUT_COST_PER_1K = 0.000600  # $0.600 per 1M output tokens

PIPELINE_STAGES = {
    "research":       {"input_tokens": 800,  "output_tokens": 500},
    "lead_scoring":   {"input_tokens": 400,  "output_tokens": 150},
    "email_gen":      {"input_tokens": 1200, "output_tokens": 400},
    "classification": {"input_tokens": 900,  "output_tokens": 50},
    "followup":       {"input_tokens": 600,  "output_tokens": 300},
}

def cost_per_prospect(stages: dict[str, dict]) -> dict:
    """Calculate total API cost per prospect across all pipeline stages."""
    total_input = sum(s["input_tokens"] for s in stages.values())
    total_output = sum(s["output_tokens"] for s in stages.values())
    cost_in = total_input / 1000 * INPUT_COST_PER_1K
    cost_out = total_output / 1000 * OUTPUT_COST_PER_1K
    return {
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "cost_input": round(cost_in, 6),
        "cost_output": round(cost_out, 6),
        "cost_total": round(cost_in + cost_out, 6),
    }

cpp = cost_per_prospect(PIPELINE_STAGES)

print("Cost per prospect (GPT-4o-mini):")
print(f"  Input tokens : {cpp['total_input_tokens']:>6,}  (${cpp['cost_input']:.4f})")
print(f"  Output tokens: {cpp['total_output_tokens']:>6,}  (${cpp['cost_output']:.4f})")
print(f"  Total cost   : ${cpp['cost_total']:.4f}")

# Time savings vs manual SDR
MANUAL_MINUTES = {
    "research": 25,
    "email_writing": 12,
    "response_handling": 8,
    "data_entry": 5,
}
AI_MINUTES = {
    "research": 0.5,
    "email_writing": 0.3,
    "response_handling": 0.2,
    "data_entry": 0.0,
}

print(f"\n{'Task':<22s} {'Manual (min)':>12s} {'AI (min)':>10s} {'Saved':>8s}")
print("-" * 55)
total_manual, total_ai = 0.0, 0.0
for task in MANUAL_MINUTES:
    m = MANUAL_MINUTES[task]
    a = AI_MINUTES[task]
    total_manual += m
    total_ai += a
    print(f"{task:<22s} {m:>12.1f} {a:>10.1f} {m - a:>7.1f}m")
print(f"{'TOTAL':<22s} {total_manual:>12.1f} {total_ai:>10.1f} {total_manual - total_ai:>7.1f}m")
print(f"\nTime savings: {(1 - total_ai / total_manual) * 100:.0f}% per prospect")

# ROI calculation
PROSPECTS_PER_MONTH = 500
SDR_ANNUAL_COST = 85_000
SDR_MONTHLY = SDR_ANNUAL_COST / 12
AI_MONTHLY = PROSPECTS_PER_MONTH * cpp["cost_total"]
HOURS_SAVED = PROSPECTS_PER_MONTH * (total_manual - total_ai) / 60
REPLY_RATE_LIFT = 0.03   # +3% absolute lift from better personalization
AVG_DEAL_VALUE = 25_000
PIPELINE_GENERATED = PROSPECTS_PER_MONTH * REPLY_RATE_LIFT * 0.20 * AVG_DEAL_VALUE

roi_table = pd.DataFrame([
    {"Metric": "Prospects / month", "Value": f"{PROSPECTS_PER_MONTH:,}"},
    {"Metric": "AI cost / month", "Value": f"${AI_MONTHLY:,.2f}"},
    {"Metric": "SDR hours saved / month", "Value": f"{HOURS_SAVED:,.0f} hrs"},
    {"Metric": "Reply rate lift", "Value": f"+{REPLY_RATE_LIFT:.1%} absolute"},
    {"Metric": "Incremental pipeline / month", "Value": f"${PIPELINE_GENERATED:,.0f}"},
    {"Metric": "Monthly ROI", "Value": f"{PIPELINE_GENERATED / AI_MONTHLY:,.0f}x"},
])
print(f"\nROI Summary:")
print(roi_table.to_string(index=False))

## Exercises

1. **Inter-rater reliability** — Run the LLM-as-judge evaluation three times with `temperature=0.3` and compute the standard deviation of each dimension's scores across runs. Which dimension has the highest variance? Add a "human baseline" by scoring 3 emails yourself and comparing.

2. **Sensitivity analysis on sample size** — Modify `required_sample_size` to sweep over baseline rates from 1 % to 10 % at a fixed MDE of 1 %. Plot the required N for each baseline and explain why lower baselines need more samples.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | LLM-as-judge provides scalable email quality scoring, but should be calibrated against human ratings. |
| 2 | Personalization depth matters more than breadth — one deep hook outperforms three surface-level mentions. |
| 3 | A/B testing outreach requires thousands of sends per variant; plan sample sizes before launching. |
| 4 | Few-shot response classification achieves high accuracy, but edge cases (forwarding, partial interest) need monitoring. |
| 5 | The full pipeline costs ~$0.001 per prospect with GPT-4o-mini — orders of magnitude cheaper than manual research. |
| 6 | ROI is driven by reply-rate lift × deal value, not by cost savings alone. |

## What's Next

- **Review**: [02_build.ipynb](./02_build.ipynb) — Revisit the pipeline to tune components based on evaluation results
- **Related**: [00_first_principles.ipynb](./00_first_principles.ipynb) — Refresh the economics model with real cost data from this evaluation

## References & Further Reading

1. [Zheng et al., "Judging LLM-as-a-Judge," NeurIPS 2023](https://arxiv.org/abs/2306.05685) — The foundational paper on using LLMs as scalable evaluators
2. [Evan Miller, "Sample Size Calculator"](https://www.evanmiller.org/ab-testing/sample-size.html) — Interactive A/B test power analysis reference
3. [OpenAI, "Pricing," 2024](https://openai.com/pricing) — Token pricing used in the cost model
4. [Gong Labs, "Cold Email Response Rates," 2023](https://www.gong.io/blog/cold-email-response-rates/) — Response rate benchmarks by personalization level